# UABR-SLM · Notebook 05 — Mixed training (Q&A + evidence reading) and G1 v3

**Phase 05** (PLAN §16–17). GPU **T4 ×2** · ~60–80 min · output ~1.5 GB.

**Why (team decision 10 Nov 2026):** after re-injection the model completes its injected fact sentences 96 % of the
time at dose 256, but cannot extract the facts from a question: the subject-specific ranking gain was only
+3.7 pp [−4.0, +11.6]. Physics of LMs 3.1 shows that extraction needs Q&A training on *some* entities, and transfers
best when it is **mixed** with the fact text rather than added afterwards.

**Design (team decisions):**
- In-main **train** questions are split (grouped by subject, stratified by relation × dose) into
  **half A** — trained as closed-book Q&A **and** RAFT-style evidence reading — and **half B** — never trained as Q&A;
  its questions supply routing labels later (avoids v1's stale-label problem). **Validation and test are untouched.**
- Formats (loss only on the answer + `<EOS>`; never on the question, evidence or action token):
  `<USER> q <ASSISTANT><ANSWER> a<EOS>` and `<USER> q <ASSISTANT><CALL><EVIDENCE> w1\nw2\nw3</EVIDENCE> a<EOS>`.
- **Two arms, same Q&A data** (team chose to compare):
  **M (mixed)** = base model (03) trained on 03b's injection stream **with** the half-A Q&A shuffled in;
  **S (sequential)** = 03b model trained on the half-A Q&A **plus** replayed pretraining text.
- **G1 v3** is measured on validation facts never seen as Q&A, with **subject-specific** checks
  (the same question with the subject replaced by "the subject"), fixing notebook 04's G1b confound.

**My defaults (change in the config cell):** 8 passes over half A; RAFT: with prob. 0.5 a supporting window from the
top-20 is swapped into the evidence when the realistic top-3 has none; arm S replay = 3× the Q&A items;
lr 2e-4 → 2e-5 (cosine, 50 warm-up) for both arms; arm selection rule fixed in advance: higher subject-specific
validation EM wins (tie → M).

**Inputs:** outputs of notebooks **00, 01, 02, 03, 03b**. **Kaggle:** Accelerator **GPU T4 ×2** · Internet **On** · Save & Run All.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.data.popqa import split_train_halves
from uabr.eval import prompts as PR
from uabr.eval.generate import load_model, generate, seed_for
from uabr.eval.metrics import alias_em, alias_f1
from uabr.eval.probes import score_continuations, candidate_sets
from uabr.eval.stats import clustered_bootstrap_mean, clustered_bootstrap_diff
from uabr.retrieval.select import select_non_overlapping

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/05_mixed" if ON_KAGGLE else os.path.abspath("outputs/05_mixed")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4 x2"

QA_EPOCHS = int(os.environ.get("QA_EPOCHS", 8))
P_ORACLE, SEED, SEQ = 0.5, 505, 512
S_REPLAY_X = 3
LR, MIN_LR, WARMUP = 2e-4, 2e-5, int(os.environ.get("WARMUP_STEPS", 50))
K, TEMP, MAX_NEW, N_DISTRACT, MIN_GAIN_PP = 8, 0.7, 16, 30, 5.0
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", 0)); HALF_SAMPLE = int(os.environ.get("HALF_SAMPLE", 500))
DOSES = ["d0", "d16", "d64", "d256"]

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True)
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
TRAIN_BIN = find_input("01_corpus/pretrain/train.bin", "TRAIN_BIN"); VAL_BIN = find_input("01_corpus/pretrain/val.bin", "VAL_BIN")
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
BASE = find_input("03_pretrain/model_final.pt", "MODEL")
REINJ = find_input("03b_reinject/model_final.pt", "MODEL_REINJ")
REINJ_BIN = find_input("03b_reinject/reinject.bin", "REINJ_BIN")
facts = pd.read_parquet(find_input("03b_reinject/facts_doses.parquet", "FACTS"), columns=["id", "dose_group", "dose"])
popqa = popqa.merge(facts, on="id", how="left"); popqa["dose_group"] = popqa["dose_group"].fillna("not_in_main")

WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
SUPPORT = {q: [int(d) for d, a, s in zip(ds, ai, si) if a and s] for q, ds, ai, si in zip(top["id"], top["doc_ids"], top["answer_in"], top["subject_in"])}
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
print(f"device {DEV} | QA epochs {QA_EPOCHS} | base {BASE} | reinjected {REINJ}")

## 3 · Split in-main train into half A (Q&A + reading) and half B (never Q&A)

In [ ]:
popqa["half"] = split_train_halves(popqa, seed=SEED)
A = popqa[popqa["half"] == "A"].reset_index(drop=True); B = popqa[popqa["half"] == "B"].reset_index(drop=True)
assert not set(A["group_id"]) & set(B["group_id"]), "subject groups must not straddle halves"
os.makedirs(f"{OUT}/manifests", exist_ok=True)
A["id"].sort_values().to_csv(f"{OUT}/manifests/halfA_ids.txt", index=False, header=False)
B["id"].sort_values().to_csv(f"{OUT}/manifests/halfB_ids.txt", index=False, header=False)
print(f"half A {len(A):,} | half B {len(B):,} | val/test untouched")
display(pd.crosstab(popqa.loc[popqa.half != "", "dose_group"], popqa.loc[popqa.half != "", "half"]))

## 4 · Build half-A training examples (closed-book + RAFT evidence), packed into 512-token items

In [ ]:
rng = np.random.default_rng(SEED)
examples, n_oracle, n_win = [], 0, []
for ep in range(QA_EPOCHS):
    for r in A.itertuples(index=False):
        examples.append(Q.closed_example(tok, r.question, r.answer))
        sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
        sup = [d for d in SUPPORT.get(r.id, []) if d not in sel]
        if sup and not any(d in SUPPORT.get(r.id, []) for d in sel) and rng.random() < P_ORACLE and sel:
            sel = list(sel); sel[int(rng.integers(0, len(sel)))] = sup[0]; n_oracle += 1
        ex, k = Q.evidence_example(tok, r.question, [WTEXT[d] for d in sel], r.answer, SEQ)
        examples.append(ex); n_win.append(k)
order = rng.permutation(len(examples))
X_qa, M_qa = Q.pack([examples[i] for i in order], SEQ)
print(f"{len(examples):,} examples -> {len(X_qa):,} packed items ({X_qa.size/1e6:.1f}M tokens, {M_qa.sum()/1e6:.2f}M targets)")
print(f"RAFT oracle swaps: {n_oracle:,} | evidence windows used: {pd.Series(n_win).value_counts().sort_index().to_dict()}")

## 5 · Item files for the two arms

In [ ]:
X_inj, M_inj = Q.lm_blocks(np.memmap(REINJ_BIN, dtype=np.uint16, mode="r"), SEQ)
X_rep, M_rep = Q.lm_blocks(np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r"), SEQ, n=S_REPLAY_X * len(X_qa), rng=rng)
ARMS = {"M": (np.concatenate([X_inj, X_qa]), np.concatenate([M_inj, M_qa]), BASE),
        "S": (np.concatenate([X_rep, X_qa]), np.concatenate([M_rep, M_qa]), REINJ)}
for name, (X, M, init) in ARMS.items():
    os.makedirs(f"{OUT}/arm_{name}", exist_ok=True)
    np.save(f"{OUT}/arm_{name}/items_x.npy", X); np.save(f"{OUT}/arm_{name}/items_m.npy", M)
    print(f"arm {name}: {len(X):,} items = {X.size/1e6:.0f}M tokens (Q&A items {len(X_qa)/len(X):.1%}) | init {os.path.basename(os.path.dirname(init))}")
del X_inj, M_inj, X_rep, M_rep

## 6 · Train both arms

In [ ]:
NPROC = max(1, torch.cuda.device_count())
env = {**os.environ, "PYTHONPATH": SRC + os.pathsep + os.environ.get("PYTHONPATH", ""),
       "NCCL_P2P_DISABLE": os.environ.get("NCCL_P2P_DISABLE", "1"), "OMP_NUM_THREADS": "2"}
mcfg = torch.load(BASE, map_location="cpu", weights_only=False)["model_cfg"]; mcfg.pop("max_seq_len", None)
RESULTS = {}
for name, (_, _, init) in ARMS.items():
    d = f"{OUT}/arm_{name}"
    run = dict(train_bin="", val_bin=VAL_BIN, out_dir=d, items_x=f"{d}/items_x.npy", items_m=f"{d}/items_m.npy", model=mcfg,
               seq_len=SEQ, global_batch_seqs=int(os.environ.get("GLOBAL_BATCH_SEQS", 256)), micro_batch_seqs=int(os.environ.get("MICRO_BATCH_SEQS", 32)),
               lr=LR, min_lr=MIN_LR, warmup_steps=WARMUP, weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0,
               precision="auto", compile=False, seed=1234, data_seed=9090, eval_every=int(os.environ.get("EVAL_EVERY", 200)),
               eval_tokens=int(os.environ.get("EVAL_TOKENS", 2_000_000)), log_every=20, ckpt_minutes=30, max_hours=11.0, max_steps=0, init_from=init)
    json.dump(run, open(f"{d}/run_config.json", "w"), indent=2)
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NPROC}", "-m", "uabr.train.pretrain", "--config", f"{d}/run_config.json"]
    stage(f"training arm {name}")
    with open(f"{d}/train_stdout.log", "a") as logf:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            if "step" not in line or "val_loss" in line or "finished" in line or "params" in line: print(line, end="")
            logf.write(line)
        assert p.wait() == 0, f"arm {name} failed - see {d}/train_stdout.log"
    va = pd.read_csv(f"{d}/train_log.csv").dropna(subset=["val_loss"])
    RESULTS[name] = {"summary": json.load(open(f"{d}/summary.json")), "val_loss_start": float(va.val_loss.iloc[0]), "val_loss_end": float(va.val_loss.iloc[-1])}
    os.remove(f"{d}/items_x.npy"); os.remove(f"{d}/items_m.npy")            # keep the output small
for n, r in RESULTS.items(): print(f"arm {n}: val loss {r['val_loss_start']:.4f} -> {r['val_loss_end']:.4f}")

## 7 · Evaluate 03b (no Q&A), M and S in the Q&A format — with and without the subject

In [ ]:
def clean(ids): return PR.clean_generation(tok.decode(ids))
answers_by_rel = popqa[(popqa.half == "A")].groupby("prop")["answer"].agg(lambda s: sorted(set(s))).to_dict()
for rel in popqa.prop.unique():                     # relations absent from half A fall back to all train answers
    answers_by_rel.setdefault(rel, sorted(set(popqa[(popqa.split == "train") & popqa.in_main & (popqa.prop == rel)]["answer"])) or ["unknown"])
val = popqa[(popqa.split == "val") & popqa.in_main].reset_index(drop=True)
if EVAL_LIMIT: val = val.head(EVAL_LIMIT)
EVAL = pd.concat([val.assign(eval_set="val"),
                  A.sample(min(HALF_SAMPLE, len(A)), random_state=1).assign(eval_set="halfA"),
                  B.sample(min(HALF_SAMPLE, len(B)), random_state=1).assign(eval_set="halfB")], ignore_index=True)
CANDS = candidate_sets(answers_by_rel, {q: (a, g) for q, a, g in zip(EVAL.id, EVAL.answer, EVAL.gold_aliases)}, dict(zip(EVAL.id, EVAL.prop)), N_DISTRACT, seed=SEED)
MODEL_PATHS = {"03b_noQA": REINJ, "M": f"{OUT}/arm_M/model_final.pt", "S": f"{OUT}/arm_S/model_final.pt"}

def nosubj(q, s):
    return q.replace(s, "the subject", 1) if s and s in q else None

def rank_acc(model, prompt_ids, cands):
    conts = [Q._enc(tok, " " + str(c)) + [T.EOS_ID] for c in cands]
    _, m = score_continuations(model, prompt_ids, conts, DEV)
    return float((m > m[0]).sum() == 0)

rows = []
for name, path in MODEL_PATHS.items():
    model = load_model(path, DEV)
    sub = EVAL[EVAL.eval_set == "val"] if name == "03b_noQA" else EVAL     # 03b never learned <EOS>: validation only (time)
    for i, r in enumerate(sub.itertuples(index=False)):
        rec = {"model": name, "id": r.id, "eval_set": r.eval_set}
        p = Q.closed_prompt_ids(tok, r.question)
        rec["greedy"] = clean(generate(model, p, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        rec["em"] = alias_em(rec["greedy"], r.gold_aliases); rec["acc1"] = rank_acc(model, p, CANDS[r.id])
        qn = nosubj(r.question, r.subj)
        if qn:
            pn = Q.closed_prompt_ids(tok, qn)
            rec["em_nosubj"] = alias_em(clean(generate(model, pn, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0]), r.gold_aliases)
            rec["acc1_nosubj"] = rank_acc(model, pn, CANDS[r.id])
        else:
            rec["em_nosubj"] = rec["acc1_nosubj"] = np.nan
        if r.eval_set == "val":
            sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
            pe, _ = Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)
            rec["greedy_ev"] = clean(generate(model, pe, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0]); rec["em_ev"] = alias_em(rec["greedy_ev"], r.gold_aliases)
        rows.append(rec)
        if (i + 1) % 750 == 0: stage(f"  {name} {i+1}/{len(sub)}")
    del model
E = pd.DataFrame(rows).merge(EVAL[["id", "eval_set", "prop", "pop_bin", "dose_group", "group_id", "prior_correct"]], on=["id", "eval_set"])
E["subj_em"] = E["em"] - E["em_nosubj"]; E["subj_rank"] = E["acc1"] - E["acc1_nosubj"]
summary = E.groupby(["model", "eval_set"]).agg(n=("id", "size"), EM=("em", "mean"), EM_no_subject=("em_nosubj", "mean"), subj_specific_EM=("subj_em", "mean"),
                                             rank_acc1=("acc1", "mean"), rank_no_subject=("acc1_nosubj", "mean"), EM_evidence=("em_ev", "mean"), prior=("prior_correct", "mean"))
display(summary.round(3))

## 8 · Dose response on validation facts never seen as Q&A (subject-specific)

In [ ]:
D, TABS = {}, {}
for name in MODEL_PATHS:
    v = E[(E.model == name) & (E.eval_set == "val") & E.dose_group.isin(DOSES)].dropna(subset=["subj_em"])
    TABS[name] = v.groupby("dose_group").agg(n=("id", "size"), EM=("em", "mean"), subj_specific_EM=("subj_em", "mean"),
                                            rank_acc1=("acc1", "mean"), subj_specific_rank=("subj_rank", "mean")).reindex(DOSES)
    for m in ["em", "subj_em", "acc1", "subj_rank"]:
        D[(name, m)] = clustered_bootstrap_diff(v[m], v.group_id, v.dose_group == "d256", v.dose_group == "d0")
    print(f"\n{name}"); display(TABS[name].round(3))
    print("  d256 - d0: " + " | ".join(f"{m} {D[(name, m)]['diff']*100:+.1f} [{D[(name, m)]['lo']*100:+.1f}, {D[(name, m)]['hi']*100:+.1f}]" for m in ["em", "subj_em", "acc1", "subj_rank"]) + "  (pp)")

## 9 · Choose θ0 (rule fixed in advance: higher subject-specific validation EM; tie → M)

In [ ]:
vs = {n: float(E[(E.model == n) & (E.eval_set == "val")]["subj_em"].mean()) for n in ["M", "S"]}
ARM = "M" if vs["M"] >= vs["S"] else "S"
THETA0 = MODEL_PATHS[ARM]
print(f"subject-specific val EM: M {vs['M']:.3%} | S {vs['S']:.3%} -> theta0 = arm {ARM}")
json.dump({"arm": ARM, "path_in_output": f"05_mixed/arm_{ARM}/model_final.pt", "rule": "higher val subject-specific EM, tie -> M", "scores": vs},
          open(f"{OUT}/theta0.json", "w"), indent=2)

## 10 · Pilot on θ0: k = 8 samples, b_quality, 4-way table, retrieval damage

In [ ]:
model = load_model(THETA0, DEV)
pilot = popqa[popqa.is_pilot].reset_index(drop=True)
if EVAL_LIMIT: pilot = pilot.head(min(len(pilot), EVAL_LIMIT))
prow = []
for i, r in enumerate(pilot.itertuples(index=False)):
    sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
    pa = Q.closed_prompt_ids(tok, r.question)
    pc, nw = Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)
    rec = {"id": r.id, "n_windows_used": nw}
    for cond, p in [("A", pa), ("C", pc)]:
        rec[f"greedy_{cond}"] = clean(generate(model, p, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        rec[f"samples_{cond}"] = [clean(s) for s in generate(model, p, K, MAX_NEW, TEMP, seed_for(SEED, r.id, cond), DEV, stop_ids)]
    prow.append(rec)
    if (i + 1) % 100 == 0: stage(f"  pilot {i+1}/{len(pilot)}")
res = pilot.merge(pd.DataFrame(prow), on="id")
for c in "AC":
    res[f"em_{c}"] = [alias_em(p, g) for p, g in zip(res[f"greedy_{c}"], res.gold_aliases)]
    res[f"sample_f1_{c}"] = [float(np.mean([alias_f1(s, g) for s in ss])) for ss, g in zip(res[f"samples_{c}"], res.gold_aliases)]
res["b_quality"] = res.sample_f1_C - res.sample_f1_A
b = res.b_quality
b_dist = {"P(b>0)": float((b > 0).mean()), "P(b<0)": float((b < 0).mean()), "P(b=0)": float((b == 0).mean()), "mean_b": float(b.mean())}
four = pd.crosstab(res.em_A.map({1.0: "closed correct", 0.0: "closed wrong"}), res.em_C.map({1.0: "evidence correct", 0.0: "evidence wrong"}))
damage = int(((res.em_A == 1) & (res.em_C == 0)).sum())
print(f"pilot closed EM {res.em_A.mean():.1%} | evidence EM {res.em_C.mean():.1%} | prior {res.prior_correct.mean():.1%} | damage {damage}")
print({k: round(v, 3) for k, v in b_dist.items()}); display(four)

## 11 · Save

In [ ]:
E.to_parquet(f"{OUT}/eval_all_models.parquet", index=False); res.to_parquet(f"{OUT}/pilot_theta0.parquet", index=False)
summary.to_csv(f"{OUT}/eval_summary.csv")
for n, t in TABS.items(): t.to_csv(f"{OUT}/dose_response_{n}.csv")
json.dump({"notebook": "05_mixed", "qa_epochs": QA_EPOCHS, "p_oracle": P_ORACLE, "s_replay_x": S_REPLAY_X, "lr": [LR, MIN_LR, WARMUP],
           "halves": {"A": len(A), "B": len(B)}, "arms": RESULTS, "theta0": ARM, "dose_diffs": {f"{k[0]}:{k[1]}": v for k, v in D.items()},
           "b_distribution": b_dist, "retrieval_damage": damage, "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}},
          open(f"{OUT}/meta.json", "w"), indent=2, default=str)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1e6:9.1f} MB  {os.path.relpath(f, OUT)}")

## 12 · Gate G1 v3 (θ0, validation facts never seen as Q&A)

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
v0 = E[(E.model == ARM) & (E.eval_set == "val")]
vd = v0[v0.dose_group.isin(["d16", "d64", "d256"])]
g_prior = clustered_bootstrap_mean((vd.em - vd.prior_correct) * 100, vd.group_id)
g_subj = clustered_bootstrap_mean(vd.subj_em.dropna() * 100, vd.dropna(subset=["subj_em"]).group_id)
add("G1a_dosed_val_EM_minus_prior_ge_5pp", g_prior["mean"] >= MIN_GAIN_PP and g_prior["lo"] > 0,
    f"{g_prior['mean']:+.1f} pp [CI {g_prior['lo']:+.1f}, {g_prior['hi']:+.1f}] (val, doses 16-256, n={g_prior['n']})")
add("G1a_subject_specific_EM_gt_0", g_subj["lo"] > 0, f"EM - EM(no subject) {g_subj['mean']:+.1f} pp [{g_subj['lo']:+.1f}, {g_subj['hi']:+.1f}]")
ds, dr = D[(ARM, "subj_em")], D[(ARM, "subj_rank")]
add("G1b_subject_specific_rises_with_dose", ds["lo"] > 0 or dr["lo"] > 0,
    f"d256-d0 subj EM {ds['diff']*100:+.1f} (lo {ds['lo']*100:+.1f}); subj rank {dr['diff']*100:+.1f} (lo {dr['lo']*100:+.1f}) pp")
add("G1c_b_not_almost_all_positive", b_dist["P(b>0)"] < 0.90, f"P(b>0) = {b_dist['P(b>0)']:.1%}")
add("G1d_retrieval_damage_exists", damage >= 1, f"{damage} questions")
vl = RESULTS[ARM]["val_loss_end"]
add("info_forgetting_vs_base_le_0.05", vl - 2.8968 <= 0.05 if ON_KAGGLE else True, f"theta0 val loss {vl:.4f} (base 2.8968)", "warn")
gA = float(E[(E.model == ARM) & (E.eval_set == "halfA")].em.mean()); gB = float(E[(E.model == ARM) & (E.eval_set == "halfB")].em.mean())
add("info_train_halves_EM", True, f"half A (trained) {gA:.1%} | half B (never Q&A) {gB:.1%} | val {v0.em.mean():.1%}", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "G1_v3", "status": status, "theta0_arm": ARM, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print(f"\nGATE G1 (v3): {status} | theta0 = arm {ARM}")

## Next
- **G1 PASS:** notebook 06 computes counterfactual benefit labels with θ0 (`theta0.json`) — routing labels come from **half B**
  (never trained as Q&A), validation/test are for evaluation only.
- **G1 FAIL:** compare the arms and the half-A vs half-B gap: high half-A EM but no transfer to half B / val = the model
  memorises Q&A pairs without extracting from pretraining → larger model / instructor decision.